# Regime Analytics & Signal Explorer

Deep-dive into per-regime performance statistics, forward transition probabilities,
and the posterior-weighted trading signal from the Regime Detection Engine.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.subplots as sp
from plotly.colors import qualitative

from rde.config import load_config
from rde.data import YFinanceSource
from rde.features.pipeline import FeaturePipeline
from rde.features.returns import LogReturns, SmoothedReturns
from rde.features.volatility import RollingVolatility
from rde.models.hmm import train_hmm
from rde.inference.viterbi import viterbi_decode
from rde.inference.online import OnlineDecoder
from rde.evaluation.regime_analytics import (
    compute_regime_stats, regime_stats_to_dataframe, regime_transition_table, transition_forecast
)
from rde.signals.regime_signal import RegimeSignalGenerator, RegimeSignalConfig, score_states

## Load pre-computed results

We load the BTC-USD results produced by `rde run --config configs/btc.yaml --signal`.
Re-run the CLI to regenerate with different parameters.

In [ ]:
SYMBOL = "BTC-USD"
results_dir = Path(f"../results/{SYMBOL}")

regimes_df = pd.read_parquet(results_dir / "regimes.parquet")
analytics_df = pd.read_parquet(results_dir / "regime_analytics.parquet")
signals_df   = pd.read_parquet(results_dir / "signals.parquet")

print(f"Regimes shape:   {regimes_df.shape}")
print(f"Analytics shape: {analytics_df.shape}")
print(f"Signals shape:   {signals_df.shape}")
print(f"\nRegime analytics:\n")
display(analytics_df[["label","count","weight_%","mean_ret_ann_%","vol_ann_%","sharpe_ann","max_drawdown_%"]])

## Per-regime performance — bar charts

In [ ]:
labels = analytics_df["label"].tolist()

# Diverging colors driven by sign of Sharpe
sharpe_vals = analytics_df["sharpe_ann"].tolist()
bar_colors = ["green" if s >= 0 else "red" for s in sharpe_vals]

fig = sp.make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        "Annualised Return (%)",
        "Annualised Sharpe",
        "Max Drawdown (%)",
    ],
    shared_xaxes=False,
)

fig.add_trace(
    go.Bar(
        x=labels,
        y=analytics_df["mean_ret_ann_%"].tolist(),
        marker_color=bar_colors,
        name="Ann. Return (%)",
        showlegend=False,
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Bar(
        x=labels,
        y=sharpe_vals,
        marker_color=bar_colors,
        name="Sharpe",
        showlegend=False,
    ),
    row=1, col=2,
)

# Drawdown is negative by convention — use same diverging scheme
dd_vals = analytics_df["max_drawdown_%"].tolist()
dd_colors = ["red" if d < 0 else "green" for d in dd_vals]

fig.add_trace(
    go.Bar(
        x=labels,
        y=dd_vals,
        marker_color=dd_colors,
        name="Max Drawdown (%)",
        showlegend=False,
    ),
    row=1, col=3,
)

fig.update_layout(
    title_text="BTC-USD: Per-Regime Performance",
    height=450,
    template="plotly_white",
)

fig.show()

## Signal vs price

In [ ]:
fig = sp.make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.65, 0.35],
    subplot_titles=["Close Price by Regime", "Regime Signal"],
    vertical_spacing=0.06,
)

# --- Top panel: Close price coloured by regime label ---
unique_labels = regimes_df["regime_label"].unique()
palette = qualitative.Plotly

for i, lbl in enumerate(sorted(unique_labels)):
    mask = regimes_df["regime_label"] == lbl
    sub = regimes_df.loc[mask]
    fig.add_trace(
        go.Scatter(
            x=sub.index,
            y=sub["Close"],
            mode="markers",
            marker=dict(size=2, color=palette[i % len(palette)]),
            name=lbl,
            legendgroup=lbl,
        ),
        row=1, col=1,
    )

# --- Bottom panel: signal line ---
fig.add_trace(
    go.Scatter(
        x=signals_df.index,
        y=signals_df["signal"],
        mode="lines",
        line=dict(color="royalblue", width=1),
        name="Signal",
        showlegend=True,
    ),
    row=2, col=1,
)

# Horizontal zero line
fig.add_hline(
    y=0,
    line=dict(color="black", dash="dash", width=1),
    row=2, col=1,
)

# Shaded ±0.25 band
x_vals = signals_df.index.tolist()
fig.add_trace(
    go.Scatter(
        x=x_vals + x_vals[::-1],
        y=[0.25] * len(x_vals) + [-0.25] * len(x_vals),
        fill="toself",
        fillcolor="rgba(100, 149, 237, 0.12)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=False,
        name="±0.25 band",
    ),
    row=2, col=1,
)

fig.update_yaxes(title_text="Close (USD)", row=1, col=1)
fig.update_yaxes(title_text="Signal", range=[-1.05, 1.05], row=2, col=1)
fig.update_layout(
    title_text="BTC-USD: Close Price by Regime (top) and Signal (bottom)",
    height=650,
    template="plotly_white",
    legend=dict(orientation="v", x=1.01, y=1),
)

fig.show()

## Forward transition probability heatmaps

In [ ]:
# Build empirical 1-step transition matrix from consecutive Viterbi state pairs
states_arr = regimes_df["regime"].values
unique_states = sorted(regimes_df["regime"].unique())
K = len(unique_states)
state_to_idx = {s: i for i, s in enumerate(unique_states)}

# Unique labels ordered by state index
state_labels = [
    regimes_df.loc[regimes_df["regime"] == s, "regime_label"].iloc[0]
    for s in unique_states
]

# 1-step count matrix
count_1 = np.zeros((K, K), dtype=float)
for t in range(len(states_arr) - 1):
    i = state_to_idx[states_arr[t]]
    j = state_to_idx[states_arr[t + 1]]
    count_1[i, j] += 1

# Row-normalise to get empirical transition matrix P
row_sums = count_1.sum(axis=1, keepdims=True)
P = np.where(row_sums > 0, count_1 / row_sums, 1.0 / K)

# h-step transition matrices via matrix power
def mat_power(M: np.ndarray, h: int) -> np.ndarray:
    result = np.eye(M.shape[0])
    base = M.copy()
    while h > 0:
        if h % 2 == 1:
            result = result @ base
        base = base @ base
        h //= 2
    return result

horizons = [1, 24, 168]
horizon_labels = ["h=1 (1 bar)", "h=24 (1 day)", "h=168 (1 week)"]

fig = sp.make_subplots(
    rows=1, cols=3,
    subplot_titles=horizon_labels,
    horizontal_spacing=0.12,
)

for col_idx, h in enumerate(horizons, start=1):
    Ph = mat_power(P, h)
    fig.add_trace(
        go.Heatmap(
            z=Ph,
            x=state_labels,
            y=state_labels,
            colorscale="Blues",
            zmin=0.0,
            zmax=1.0,
            text=[[f"{v:.2f}" for v in row] for row in Ph],
            texttemplate="%{text}",
            showscale=(col_idx == 3),
            colorbar=dict(title="Prob", len=0.8) if col_idx == 3 else None,
        ),
        row=1, col=col_idx,
    )

fig.update_layout(
    title_text="Empirical Transition Probabilities at h=1, 24, 168 bars",
    height=400,
    template="plotly_white",
)

# Y-axis: from state labels (row = origin state)
for col_idx in range(1, 4):
    fig.update_yaxes(title_text="From state", row=1, col=col_idx)
    fig.update_xaxes(title_text="To state", row=1, col=col_idx)

fig.show()

## Signal score vector

Each state is ranked by in-sample Sharpe ratio and mapped to a score in [-1, 1].
The signal is the posterior-weighted dot product of scores × filtered probabilities.

In [ ]:
sharpe_values = analytics_df["sharpe_ann"].values
K_states = len(sharpe_values)

# Rank-based score mapping: lowest Sharpe → -1, highest → +1
ranks = np.argsort(np.argsort(sharpe_values))  # double argsort = rank
if K_states > 1:
    scores = 2.0 * ranks / (K_states - 1) - 1.0
else:
    scores = np.zeros(K_states)

score_colors = ["green" if s >= 0 else "red" for s in scores]

fig = go.Figure(
    go.Bar(
        x=analytics_df["label"].tolist(),
        y=scores.tolist(),
        marker_color=score_colors,
        text=[f"{s:+.3f}" for s in scores],
        textposition="outside",
    )
)

fig.add_hline(y=0, line=dict(color="black", dash="dash", width=1))

fig.update_layout(
    title_text="State quality scores (Sharpe-ranked)",
    xaxis_title="State label",
    yaxis_title="Score",
    yaxis_range=[-1.2, 1.2],
    height=400,
    template="plotly_white",
)

fig.show()

## Signal autocorrelation and distribution

In [ ]:
signal_series = signals_df["signal"].dropna()
max_lag = 168

# Manual autocorrelation for lags 1..max_lag
acf_values = [
    float(signal_series.autocorr(lag=lag))
    for lag in range(1, max_lag + 1)
]
lags = list(range(1, max_lag + 1))

fig = sp.make_subplots(
    rows=1, cols=2,
    subplot_titles=["Autocorrelation (lags 1–168)", "Signal distribution"],
    column_widths=[0.6, 0.4],
)

# ACF bar chart
fig.add_trace(
    go.Bar(
        x=lags,
        y=acf_values,
        marker_color="steelblue",
        name="ACF",
        showlegend=False,
    ),
    row=1, col=1,
)

# 95% confidence band (approx ±1.96/sqrt(T))
n_obs = len(signal_series)
ci = 1.96 / np.sqrt(n_obs)
fig.add_hline(y=ci,  line=dict(color="red", dash="dot", width=1), row=1, col=1)
fig.add_hline(y=-ci, line=dict(color="red", dash="dot", width=1), row=1, col=1)
fig.add_hline(y=0,   line=dict(color="black", dash="dash", width=0.8), row=1, col=1)

# Histogram
fig.add_trace(
    go.Histogram(
        x=signal_series.tolist(),
        nbinsx=50,
        marker_color="steelblue",
        opacity=0.75,
        name="Signal",
        showlegend=False,
    ),
    row=1, col=2,
)

fig.update_xaxes(title_text="Lag (bars)", row=1, col=1)
fig.update_yaxes(title_text="ACF", row=1, col=1)
fig.update_xaxes(title_text="Signal value", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=2)

fig.update_layout(
    title_text="Signal autocorrelation (left) and distribution (right)",
    height=420,
    template="plotly_white",
)

fig.show()

## Regime dwell-time distributions

In [ ]:
# Extract run-length sequences per state from the Viterbi path
def compute_dwell_times(state_sequence: np.ndarray) -> dict:
    """Return {state_index: array_of_run_lengths} from a 1-D state sequence."""
    dwell: dict = {}
    if len(state_sequence) == 0:
        return dwell
    current = state_sequence[0]
    run = 1
    for t in range(1, len(state_sequence)):
        if state_sequence[t] == current:
            run += 1
        else:
            dwell.setdefault(current, []).append(run)
            current = state_sequence[t]
            run = 1
    dwell.setdefault(current, []).append(run)  # last run
    return {k: np.array(v) for k, v in dwell.items()}


dwell_times = compute_dwell_times(regimes_df["regime"].values)

# Build ordered label list aligned to unique_states (defined in cell-9)
# Recompute in case cell-9 was not run
_unique_states = sorted(regimes_df["regime"].unique())
_state_labels = [
    regimes_df.loc[regimes_df["regime"] == s, "regime_label"].iloc[0]
    for s in _unique_states
]
_K = len(_unique_states)

n_cols = min(_K, 3)
n_rows = (_K + n_cols - 1) // n_cols

fig = sp.make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=_state_labels,
    horizontal_spacing=0.12,
    vertical_spacing=0.15,
)

palette = qualitative.Plotly

for idx, (state, lbl) in enumerate(zip(_unique_states, _state_labels)):
    row = idx // n_cols + 1
    col = idx % n_cols + 1
    runs = dwell_times.get(state, np.array([]))
    mean_dwell = float(runs.mean()) if len(runs) > 0 else float("nan")

    fig.add_trace(
        go.Histogram(
            x=runs.tolist(),
            nbinsx=40,
            marker_color=palette[idx % len(palette)],
            opacity=0.75,
            name=lbl,
            showlegend=False,
        ),
        row=row, col=col,
    )

    if not np.isnan(mean_dwell):
        fig.add_vline(
            x=mean_dwell,
            line=dict(color="black", dash="dash", width=1.5),
            annotation_text=f"mean={mean_dwell:.1f}h",
            annotation_position="top right",
            row=row, col=col,
        )

    fig.update_xaxes(title_text="Dwell time (bars)", row=row, col=col)
    fig.update_yaxes(title_text="Count", row=row, col=col)

fig.update_layout(
    title_text="Dwell-time distributions per state",
    height=320 * n_rows,
    template="plotly_white",
)

fig.show()